In [18]:
# ============================================================
# INDICE DI OVERTOURISM - Composizione finale (0-1)
# ============================================================
# PASSO 1: caricamento e unione dei 4 indicatori dal database
#
# NOTA IMPORTANTE SULLA COPERTURA DEI DATI:
# - Densita' Ricettiva e Utilizzazione Lorda sono costruite su uno
#   SCAFFOLD PIENO: tutti i 377 comuni x 12 mesi, sempre presenti,
#   anche quando letti_totali = 0 (nessuna offerta ricettiva -> zero
#   vero, gia' gestito nei rispettivi indicatori).
# - Densita' Turistica e Intensita' Turistica, invece, esistono SOLO
#   per i comuni con presenze turistiche effettivamente registrate
#   (315 comuni su 377 nel 2025: i restanti 62 comuni non hanno
#   nessun dato di presenze in questa fonte, non uno zero).
#
# Di conseguenza il join finale deve partire dallo SCAFFOLD PIENO a
# 377 comuni (preso da Densita' Ricettiva), e i comuni senza turismo
# registrato devono risultare ESCLUSI DAL CALCOLO su Densita' Turistica
# e Intensita' Turistica (valore mancante, non zero) - coerente con la
# decisione presa: per questi due indicatori un dato mancante non va
# confuso con "zero turismo", mentre per Ricettiva/Utilizzazione lo zero
# e' un dato reale e viene trattato come tale.

# PERCHE' LA MEDIA GEOMETRICA per calcolo OverTourism
# Penalizza gli squilibri tra i 4 indicatori. Un comune alto su 3
# indicatori ma quasi zero sul quarto ottiene un indice molto piu'
# basso della media aritmetica degli stessi 4 valori - coerente con
# l'idea che l'overtourism "equilibrato" su piu' dimensioni sia un
# segnale piu' forte di uno squilibrato su una sola.
#
# GESTIONE DEGLI ZERI E DEI DATI MANCANTI:
# - Densita' Ricettiva: lo zero e' un dato VERO (nessuna offerta),
#   sempre incluso. Se e' zero, l'intero indice diventa zero (e' il
#   comportamento naturale della media geometrica: un fattore zero
#   annulla il prodotto). Verificato: non distorce nulla, perche' i
#   comuni senza letti in questi dati hanno sempre anche zero
#   presenze turistiche registrate (Densita'/Intensita' gia' escluse).
# - Densita' Turistica, Intensita' Turistica, Utilizzazione Lorda:
#   quando il dato non e' affidabile (copertura_sufficiente=False),
#   la riga viene ESCLUSA dal calcolo per quell'indicatore (NaN),
#   non forzata a zero. La media geometrica si ricalcola solo sugli
#   indicatori effettivamente disponibili per quella riga.
# - Se per una riga NESSUN indicatore e' disponibile, l'indice finale
#   e' NaN (non calcolabile), non zero.
#
# ============================================================
from pathlib import Path
import pandas as pd
import numpy as np
import duckdb

BASE_DIR = Path(r"../")
DB_DIR = BASE_DIR / "db"
DB_PATH = DB_DIR / "sardegna_overtourism.duckdb"

con = duckdb.connect(str(DB_PATH))

print("Tabelle utilizzate:")
print("  presentation.densita_turistica")
print("  presentation.densita_ricettiva")
print("  presentation.intensita_turistica")
print("  presentation.utilizzazione_lorda")

Tabelle utilizzate:
  presentation.densita_turistica
  presentation.densita_ricettiva
  presentation.intensita_turistica
  presentation.utilizzazione_lorda


In [19]:
# --- carico i 4 indicatori dal database ---
dens_tur = con.execute("""
    SELECT comune, anno, mese, densita_turistica, copertura_sufficiente AS ok_turistica
    FROM presentation.densita_turistica
""").df()

dens_ric = con.execute("""
    SELECT comune, anno, mese, densita_ricettiva, copertura_sufficiente AS ok_ricettiva
    FROM presentation.densita_ricettiva
""").df()

intensita = con.execute("""
    SELECT comune, anno, mese, intensita_turistica_presenze, copertura_sufficiente AS ok_intensita
    FROM presentation.intensita_turistica
""").df()

util_lorda = con.execute("""
    SELECT comune, anno, mese, utilizzazione_lorda_pct, copertura_sufficiente AS ok_utilizzazione
    FROM presentation.utilizzazione_lorda
""").df()

print(f"\nRighe caricate:")
print(f"  densita_turistica:     {len(dens_tur)} righe | {dens_tur['comune'].nunique()} comuni")
print(f"  densita_ricettiva:     {len(dens_ric)} righe | {dens_ric['comune'].nunique()} comuni")
print(f"  intensita_turistica:   {len(intensita)} righe | {intensita['comune'].nunique()} comuni")
print(f"  utilizzazione_lorda:   {len(util_lorda)} righe | {util_lorda['comune'].nunique()} comuni")


Righe caricate:
  densita_turistica:     13032 righe | 330 comuni
  densita_ricettiva:     18096 righe | 377 comuni
  intensita_turistica:   12984 righe | 330 comuni
  utilizzazione_lorda:   16224 righe | 354 comuni


In [20]:
# --- scaffold: parto dalla base piena (Densita' Ricettiva copre tutti i comuni) ---
# NON uso Densita' Turistica come base, perche' le mancherebbero i 62 comuni
# senza turismo registrato: userei una base incompleta fin dall'inizio.

base = dens_ric[["comune", "anno", "mese"]].drop_duplicates()
print(f"\nScaffold base (da Densita' Ricettiva, copertura piena): {len(base)} righe")

# join di tutti e 4 gli indicatori sullo scaffold pieno.
# Densita' Turistica e Intensita' Turistica avranno valori NaN per i
# comuni senza presenze registrate: questo e' VOLUTO, rappresenta un
# dato mancante da escludere, non uno zero.
indicatori = (
    base
    .merge(dens_tur, on=["comune", "anno", "mese"], how="left")
    .merge(dens_ric, on=["comune", "anno", "mese"], how="left")
    .merge(intensita, on=["comune", "anno", "mese"], how="left")
    .merge(util_lorda, on=["comune", "anno", "mese"], how="left")
)

print(f"\nDopo il join: {len(indicatori)} righe")
print(f"  righe senza Densita' Turistica (comune senza turismo registrato): "
      f"{indicatori['densita_turistica'].isna().sum()}")
print(f"  righe senza Intensita' Turistica: "
      f"{indicatori['intensita_turistica_presenze'].isna().sum()}")


Scaffold base (da Densita' Ricettiva, copertura piena): 18096 righe

Dopo il join: 18096 righe
  righe senza Densita' Turistica (comune senza turismo registrato): 5112
  righe senza Intensita' Turistica: 5112


In [21]:
print("\n--- Verifica Cagliari 2025 (prime 3 righe) ---")
print(indicatori[(indicatori["comune"] == "Cagliari") & (indicatori["anno"] == 2025)]
      .head(3).to_string(index=False))


--- Verifica Cagliari 2025 (prime 3 righe) ---
  comune  anno  mese  densita_turistica ok_turistica  densita_ricettiva  ok_ricettiva  intensita_turistica_presenze ok_intensita  utilizzazione_lorda_pct ok_utilizzazione
Cagliari  2025     1             418.33         True             202.73          True                        0.2422         True                    17.88             True
Cagliari  2025     2             470.79         True             202.73          True                        0.2725         True                    17.88             True
Cagliari  2025     3             586.51         True             202.73          True                        0.3395         True                    17.88             True


In [22]:
print("\n--- Verifica un comune senza turismo registrato (es. Bidonì) ---")
print(indicatori[(indicatori["comune"] == "Bidonì") & (indicatori["anno"] == 2025)]
      .head(3).to_string(index=False))


--- Verifica un comune senza turismo registrato (es. Bidonì) ---
comune  anno  mese  densita_turistica ok_turistica  densita_ricettiva  ok_ricettiva  intensita_turistica_presenze ok_intensita  utilizzazione_lorda_pct ok_utilizzazione
Bidonì  2025     1                NaN          NaN                0.0         False                           NaN          NaN                      NaN              NaN
Bidonì  2025     2                NaN          NaN                0.0         False                           NaN          NaN                      NaN              NaN
Bidonì  2025     3                NaN          NaN                0.0         False                           NaN          NaN                      NaN              NaN


In [23]:
# ============================================================
# PASSO 2: calcolo ancore di normalizzazione (winsor p99 + min-max)
# e applicazione ai 4 indicatori
#
# Le ancore (min, cap) vengono SALVATE in una tabella dedicata, cosi'
# restano fisse e riutilizzabili quando in futuro si aggiungeranno
# nuovi anni: senza congelarle, l'indice non sarebbe piu' confrontabile
# nel tempo (lo stesso comune-mese potrebbe cambiare punteggio solo
# perche' e' cambiata la scala, non i suoi dati).
# ============================================================

def normalizza_winsor_minmax(serie):
    """Winsorizzazione al 99esimo percentile + min-max, come validato mesi fa."""
    minimo = serie.min()
    cap = serie.quantile(0.99)
    capped = serie.clip(upper=cap)
    norm = (capped - minimo) / (cap - minimo)
    return norm.round(4), minimo, cap

ancore = []

# --- Densita' Turistica e Intensita' Turistica: ancore SOLO sui dati affidabili ---
norm_dt, min_dt, cap_dt = normalizza_winsor_minmax(
    indicatori.loc[indicatori["ok_turistica"] == True, "densita_turistica"]
)
norm_it, min_it, cap_it = normalizza_winsor_minmax(
    indicatori.loc[indicatori["ok_intensita"] == True, "intensita_turistica_presenze"]
)

# --- Densita' Ricettiva e Utilizzazione Lorda: ancore su TUTTE le righe ---
# (lo zero e' un dato vero, fa parte della distribuzione, non va escluso)
norm_dr, min_dr, cap_dr = normalizza_winsor_minmax(indicatori["densita_ricettiva"])
norm_ul, min_ul, cap_ul = normalizza_winsor_minmax(indicatori["utilizzazione_lorda_pct"])

ancore = pd.DataFrame([
    {"indicatore": "densita_turistica", "ancora_min": min_dt, "ancora_cap_p99": cap_dt},
    {"indicatore": "intensita_turistica", "ancora_min": min_it, "ancora_cap_p99": cap_it},
    {"indicatore": "densita_ricettiva", "ancora_min": min_dr, "ancora_cap_p99": cap_dr},
    {"indicatore": "utilizzazione_lorda", "ancora_min": min_ul, "ancora_cap_p99": cap_ul},
])
print("=== Ancore di normalizzazione (salvate per riuso futuro) ===")
print(ancore.to_string(index=False))

=== Ancore di normalizzazione (salvate per riuso futuro) ===
         indicatore  ancora_min  ancora_cap_p99
  densita_turistica         0.0     1710.440400
intensita_turistica         0.0       30.369569
  densita_ricettiva         0.0      204.820000
utilizzazione_lorda         0.0       29.620000


In [24]:
# --- applico la normalizzazione con le ancore calcolate sopra ---
# per Densita' Turistica e Intensita': applico la formula a tutti,
# ma poi azzero (NaN) dove il dato non era affidabile -> escluso dal calcolo
cap_series_dt = indicatori["densita_turistica"].clip(upper=cap_dt)
indicatori["densita_turistica_norm"] = ((cap_series_dt - min_dt) / (cap_dt - min_dt)).round(4)
indicatori.loc[indicatori["ok_turistica"] != True, "densita_turistica_norm"] = np.nan

cap_series_it = indicatori["intensita_turistica_presenze"].clip(upper=cap_it)
indicatori["intensita_turistica_norm"] = ((cap_series_it - min_it) / (cap_it - min_it)).round(4)
indicatori.loc[indicatori["ok_intensita"] != True, "intensita_turistica_norm"] = np.nan

# per Densita' Ricettiva e Utilizzazione: applico a tutti, nessuna esclusione
# (lo zero vero resta zero vero anche dopo la normalizzazione)
cap_series_dr = indicatori["densita_ricettiva"].clip(upper=cap_dr)
indicatori["densita_ricettiva_norm"] = ((cap_series_dr - min_dr) / (cap_dr - min_dr)).round(4)

cap_series_ul = indicatori["utilizzazione_lorda_pct"].clip(upper=cap_ul)
indicatori["utilizzazione_lorda_norm"] = ((cap_series_ul - min_ul) / (cap_ul - min_ul)).round(4)

print("\n--- Verifica Cagliari 2025 (agosto, mese di picco) ---")
r = indicatori[(indicatori["comune"] == "Cagliari") & (indicatori["anno"] == 2025) & (indicatori["mese"] == 8)]
print(r[["comune", "mese", "densita_turistica", "densita_turistica_norm",
         "intensita_turistica_presenze", "intensita_turistica_norm",
         "densita_ricettiva", "densita_ricettiva_norm",
         "utilizzazione_lorda_pct", "utilizzazione_lorda_norm"]].to_string(index=False))

print("\n--- Verifica Bidonì 2025 (comune senza turismo/offerta) ---")
r = indicatori[(indicatori["comune"] == "Bidonì") & (indicatori["anno"] == 2025) & (indicatori["mese"] == 8)]
print(r[["comune", "mese", "densita_turistica", "densita_turistica_norm",
         "intensita_turistica_presenze", "intensita_turistica_norm",
         "densita_ricettiva", "densita_ricettiva_norm",
         "utilizzazione_lorda_pct", "utilizzazione_lorda_norm"]].to_string(index=False))


--- Verifica Cagliari 2025 (agosto, mese di picco) ---
  comune  mese  densita_turistica  densita_turistica_norm  intensita_turistica_presenze  intensita_turistica_norm  densita_ricettiva  densita_ricettiva_norm  utilizzazione_lorda_pct  utilizzazione_lorda_norm
Cagliari     8            2068.53                     1.0                        1.1974                    0.0394             202.73                  0.9898                    17.88                    0.6036

--- Verifica Bidonì 2025 (comune senza turismo/offerta) ---
comune  mese  densita_turistica  densita_turistica_norm  intensita_turistica_presenze  intensita_turistica_norm  densita_ricettiva  densita_ricettiva_norm  utilizzazione_lorda_pct  utilizzazione_lorda_norm
Bidonì     8                NaN                     NaN                           NaN                       NaN                0.0                     0.0                      NaN                       NaN


In [25]:
# ============================================================
# PASSO 3: composizione finale - media geometrica dei 4 indicatori
# normalizzati, con esclusione riga per riga dei valori mancanti
#
# La media geometrica penalizza gli squilibri: un comune alto su 3
# indicatori ma quasi zero sul quarto ottiene un indice molto piu'
# basso della media aritmetica degli stessi 4 valori - coerente con
# l'idea che l'overtourism "equilibrato" su piu' dimensioni e' un
# segnale piu' forte di uno squilibrato su una sola.
# ============================================================

def media_geometrica_riga(valori):
    """Media geometrica su una lista di valori, escludendo i NaN.
    Se tra i valori disponibili c'e' uno zero vero, il risultato e' 0
    (comportamento naturale della media geometrica: un solo fattore
    zero annulla il prodotto). Se non c'e' NESSUN valore disponibile,
    il risultato e' NaN (indice non calcolabile per quella riga)."""
    disponibili = [v for v in valori if pd.notna(v)]
    if len(disponibili) == 0:
        return np.nan, 0
    if any(v == 0 for v in disponibili):
        return 0.0, len(disponibili)
    prodotto = np.prod(disponibili)
    return round(prodotto ** (1 / len(disponibili)), 4), len(disponibili)

risultati = indicatori.apply(
    lambda r: media_geometrica_riga([
        r["densita_turistica_norm"],
        r["intensita_turistica_norm"],
        r["densita_ricettiva_norm"],
        r["utilizzazione_lorda_norm"],
    ]),
    axis=1, result_type="expand"
)
indicatori["indice_overtourism"] = risultati[0]
indicatori["n_indicatori_disponibili"] = risultati[1]

print("=== Distribuzione dell'indice di overtourism ===")
print(indicatori["indice_overtourism"].describe())

print(f"\nRighe con indice non calcolabile (0 indicatori disponibili): "
      f"{indicatori['indice_overtourism'].isna().sum()}")
print("\nDistribuzione per numero di indicatori disponibili:")
print(indicatori["n_indicatori_disponibili"].value_counts().sort_index())

=== Distribuzione dell'indice di overtourism ===
count    18096.000000
mean         0.032819
std          0.101708
min          0.000000
25%          0.000000
50%          0.003200
75%          0.016600
max          0.941600
Name: indice_overtourism, dtype: float64

Righe con indice non calcolabile (0 indicatori disponibili): 0

Distribuzione per numero di indicatori disponibili:
n_indicatori_disponibili
1.0     1896
2.0     3438
3.0       12
4.0    12750
Name: count, dtype: int64


In [26]:
print("\n--- Verifica Cagliari 2025, agosto ---")
r = indicatori[(indicatori["comune"] == "Cagliari") & (indicatori["anno"] == 2025) & (indicatori["mese"] == 8)]
print(r[["comune", "mese", "densita_turistica_norm", "intensita_turistica_norm",
         "densita_ricettiva_norm", "utilizzazione_lorda_norm",
         "indice_overtourism", "n_indicatori_disponibili"]].to_string(index=False))


--- Verifica Cagliari 2025, agosto ---
  comune  mese  densita_turistica_norm  intensita_turistica_norm  densita_ricettiva_norm  utilizzazione_lorda_norm  indice_overtourism  n_indicatori_disponibili
Cagliari     8                     1.0                    0.0394                  0.9898                    0.6036              0.3917                       4.0


In [27]:
print("\n--- Verifica Bidonì 2025, agosto ---")
r = indicatori[(indicatori["comune"] == "Bidonì") & (indicatori["anno"] == 2025) & (indicatori["mese"] == 8)]
print(r[["comune", "mese", "densita_turistica_norm", "intensita_turistica_norm",
         "densita_ricettiva_norm", "utilizzazione_lorda_norm",
         "indice_overtourism", "n_indicatori_disponibili"]].to_string(index=False))


--- Verifica Bidonì 2025, agosto ---
comune  mese  densita_turistica_norm  intensita_turistica_norm  densita_ricettiva_norm  utilizzazione_lorda_norm  indice_overtourism  n_indicatori_disponibili
Bidonì     8                     NaN                       NaN                     0.0                       NaN                 0.0                       1.0


In [28]:
print("\n--- Verifica Villasimius 2025, agosto ---")
r = indicatori[(indicatori["comune"] == "Villasimius") & (indicatori["anno"] == 2025) & (indicatori["mese"] == 8)]
print(r[["comune", "mese", "densita_turistica_norm", "intensita_turistica_norm",
         "densita_ricettiva_norm", "utilizzazione_lorda_norm",
         "indice_overtourism", "n_indicatori_disponibili"]].to_string(index=False))


--- Verifica Villasimius 2025, agosto ---
     comune  mese  densita_turistica_norm  intensita_turistica_norm  densita_ricettiva_norm  utilizzazione_lorda_norm  indice_overtourism  n_indicatori_disponibili
Villasimius     8                     1.0                       1.0                     1.0                    0.4976              0.8399                       4.0


In [29]:
print("\n--- Verifica Villasimius 2025, gennaio ---")
r = indicatori[(indicatori["comune"] == "Villasimius") & (indicatori["anno"] == 2025) & (indicatori["mese"] == 1)]
print(r[["comune", "mese", "densita_turistica_norm", "intensita_turistica_norm",
         "densita_ricettiva_norm", "utilizzazione_lorda_norm",
         "indice_overtourism", "n_indicatori_disponibili"]].to_string(index=False))


--- Verifica Villasimius 2025, gennaio ---
     comune  mese  densita_turistica_norm  intensita_turistica_norm  densita_ricettiva_norm  utilizzazione_lorda_norm  indice_overtourism  n_indicatori_disponibili
Villasimius     1                  0.0058                    0.0051                     1.0                    0.4976              0.0619                       4.0


In [30]:
print("\n--- Top 10 comuni-mese per indice di overtourism (2025) ---")
top = indicatori[indicatori["anno"] == 2025].nlargest(10, "indice_overtourism")
print(top[["comune", "mese", "indice_overtourism", "n_indicatori_disponibili"]].to_string(index=False))


--- Top 10 comuni-mese per indice di overtourism (2025) ---
 comune  mese  indice_overtourism  n_indicatori_disponibili
Tortolì     8              0.9416                       4.0
Tortolì     7              0.8830                       4.0
 Budoni     6              0.8630                       4.0
 Budoni     7              0.8630                       4.0
 Budoni     8              0.8630                       4.0
 Badesi     6              0.8616                       4.0
 Badesi     7              0.8616                       4.0
 Badesi     8              0.8616                       4.0
 Badesi     9              0.8616                       4.0
 Orosei     7              0.8486                       4.0


# NOTA INTERPRETATIVA IMPORTANTE (caso Villasimius):
L'Utilizzazione Lorda e' un valore ANNUALE spalmato costante sui 12 mesi (letti e presenze annue non variano di mese in mese). 

Questo significa che un comune con capacita' ricettiva enorme costruita apposta per il picco estivo (es. Villasimius: 17.027 letti) puo' avere un'Utilizzazione Lorda ANNUA relativamente modesta (~15%), perche' quella capacita' resta in gran parte vuota per 10-11 mesi l'anno. Questo valore, essendo costante, si applica UGUALMENTE anche al mese di agosto (il vero picco), diluendo nella media geometrica di agosto un segnale che gli altri 3 indicatori mensili mostrano invece fortissimo. Non e' un errore: riflette il fatto che l'offerta e' dimensionata per il picco, non che il picco non esista.

In [31]:
# ============================================================
# Salvataggio CSV per anno + scrittura nel database
# ============================================================
OUT_DIR = BASE_DIR / "data" / "indice_overtourism"
OUT_DIR.mkdir(parents=True, exist_ok=True)

ANNI = sorted(indicatori["anno"].unique())

# colonne finali: dato grezzo + normalizzato per ciascun indicatore,
# piu' l'indice finale e il numero di indicatori disponibili per riga
COLONNE_OUTPUT = [
    "comune", "anno", "mese",
    "densita_turistica", "densita_turistica_norm",
    "intensita_turistica_presenze", "intensita_turistica_norm",
    "densita_ricettiva", "densita_ricettiva_norm",
    "utilizzazione_lorda_pct", "utilizzazione_lorda_norm",
    "indice_overtourism", "n_indicatori_disponibili",
]

for anno in ANNI:
    out_anno = (
        indicatori[indicatori["anno"] == anno][COLONNE_OUTPUT]
        .sort_values(["comune", "mese"])
        .reset_index(drop=True)
    )
    percorso_out = OUT_DIR / f"indice_overtourism_{anno}.csv"
    out_anno.to_csv(percorso_out, index=False, encoding="utf-8-sig")
    print(f"{anno}: {out_anno['comune'].nunique()} comuni | {len(out_anno)} righe | salvato -> {percorso_out}")

# --- scrittura nel database (tabella unica, tutti gli anni) ---
con.register("indice_overtourism_temp", indicatori[COLONNE_OUTPUT])
con.execute("""
    CREATE OR REPLACE TABLE presentation.indice_overtourism AS
    SELECT * FROM indice_overtourism_temp
""")

verifica_db = con.execute("""
    SELECT COUNT(*) AS n_righe, COUNT(DISTINCT comune) AS n_comuni, COUNT(DISTINCT anno) AS n_anni,
           ROUND(AVG(indice_overtourism), 4) AS media_indice
    FROM presentation.indice_overtourism
""").df()
print("\n=== SCRITTURA NEL DATABASE - presentation.indice_overtourism ===")
print(verifica_db.to_string(index=False))

# NUOVO - creazione df e csv annuale

out_complessivo = indicatori[COLONNE_OUTPUT].sort_values(["comune", "anno", "mese"]).reset_index(drop=True)
out_complessivo.to_csv(f"{OUT_DIR}/indice_overtourism_complessivo.csv", index = False, encoding="utf-8-sig")


2022: 377 comuni | 4524 righe | salvato -> ../data/indice_overtourism/indice_overtourism_2022.csv
2023: 377 comuni | 4524 righe | salvato -> ../data/indice_overtourism/indice_overtourism_2023.csv
2024: 377 comuni | 4524 righe | salvato -> ../data/indice_overtourism/indice_overtourism_2024.csv
2025: 377 comuni | 4524 righe | salvato -> ../data/indice_overtourism/indice_overtourism_2025.csv

=== SCRITTURA NEL DATABASE - presentation.indice_overtourism ===
 n_righe  n_comuni  n_anni  media_indice
   18096       377       4        0.0328


In [32]:
print("\n=== VERIFICA DAL DB - Top 10 comuni-mese per indice (tutti gli anni) ===")
print(con.execute("""
    SELECT comune, anno, mese, indice_overtourism
    FROM presentation.indice_overtourism
    ORDER BY indice_overtourism DESC
    LIMIT 10
""").df().to_string(index=False))


=== VERIFICA DAL DB - Top 10 comuni-mese per indice (tutti gli anni) ===
     comune  anno  mese  indice_overtourism
    Tortolì  2025     8              0.9416
    Tortolì  2025     7              0.8830
Villasimius  2022     6              0.8745
Villasimius  2022     7              0.8745
Villasimius  2022     8              0.8745
Villasimius  2022     9              0.8745
Villasimius  2023     6              0.8699
Villasimius  2023     7              0.8699
Villasimius  2023     8              0.8699
Villasimius  2023     9              0.8699


In [33]:
con.close()
print("Connessione al database chiusa.")

Connessione al database chiusa.
